In [ ]:
# W8 Day 5 - Capability 与 Connector
# matplotlib 中文字体配置
from matplotlib import font_manager
import matplotlib.pyplot as plt
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"中文字体配置完成: {font_name}")

# 🧱 LangChat 心智模型｜第8周-Day5：Capability 与 Connector

> **链路第五步：执行时怎么连到业务系统？**
>
> **日期**：2026-07-24（周五）
>
> **今日核心问题：为什么 Capability 不是 Plugin？**

## 📅 学习进度

```
W1  ████████████████████ ✅ Transformer与大模型训练
W2  ████████████████████ ✅ 微调与RLHF
W3  ████████████████████ ✅ RAG与知识增强
W4  ████████████████████ ✅ 推理与思维链
W5  ████████████████████ ✅ Agent与工具使用
W6  ████████████████████ ✅ LLM Agent实战
W7  ████████████████████ ✅ 数字员工架构深化
W8  █████████████████░░░ 🔥 LangChat端到端链路 (Day5/7)
W9  ░░░░░░░░░░░░░░░░░░░░ 📐 Domain Deep Dive
W10 ░░░░░░░░░░░░░░░░░░░░ 🛡️ Governance
W11 ░░░░░░░░░░░░░░░░░░░░ 📊 Code Reality
```

**进度: 8/13 周 (61.5%) | Day 40/91**

# 🔄 往期回顾（W8 链路前四天）

| Day | 主题 | 核心要点 | 与今天的关系 |
|-----|------|----------|--------------|
| Day1 | 用户意图 | Agent Host 直接调用 LangChat | Agent Host 要执行的"能力"是什么？ |
| Day2 | ApplicationContract | 传输无关的业务契约 | Contract 的实现需要调用底层"能力" |
| Day3 | Blueprint→ExecutionPlan | 确定性编译 | 执行计划引用"能力清单" |
| Day4 | Runtime 无状态执行 | 手术室模式 | Runtime 执行时通过什么连外部系统？ |

## 💡 今日核心链路

```
Day4 Runtime 执行 SkillRelease
    ↓
Day5 SkillRelease 引用 Capability（治理描述）
         → Connector（连接器）→ 企业系统
```

**今天回答：Capability 是什么？Connector 是什么？为什么不能合在一起？**

# 📚 Part 1：为什么 Capability 不是 Plugin？

## 生活类比：医院专科能力 vs 外包工

| 模式 | 类比 | 特点 |
|------|------|------|
| Plugin（插件） | 科室整体外包 | 自带设备、自定流程、医院不可管控 |
| Capability（能力） | 专科能力认证 | 医院定义标准、资质审核、产出规范 |

## Plugin 模式的五大问题

| 问题 | 说明 |
|------|------|
| 治理黑洞 | Plugin 内部不可见、不可审 |
| 安全绕过 | Plugin 拥有调用方全部权限 |
| 版本失控 | Plugin 升级行为变化，无版本管理 |
| 责任不清 | 出错时无法追溯 |
| 不可组合 | 无统一契约，无法被 SkillRelease 引用 |

## Capability 的三道防线

```
第一道：Capability Descriptor（描述符）
  — 声明：输入输出、scope、effect、approval_policy
  — published 后不可变（frozen=True）

第二道：SkillRelease Binding（绑定）
  — 定义：哪个能力被哪个技能使用
  — 技能是唯一可执行的业务单元

第三道：Connector（连接器）
  — 约束：只在已授权 execution context 内可用
  — 不可被 Channel/Gateway/Runtime 直接调用
```

> **一句话：Capability 定义"能做什么"，SkillRelease 定义"用它做什么"，Connector 定义"怎么连过去"。**

# 📚 Part 2：ADR 架构设计

## ADR-001 §7：三层分类法

| 概念 | 定位 | P0 角色 |
|------|------|---------|
| SkillRelease | 对外消费与发布单元 | **唯一对外执行入口** |
| Capability | 受治理的执行依赖 | Provider 契约描述 |
| Workflow | 内部执行表示 | 不对 Agent Host 暴露 |

## ADR-003（docs）：Capability × Industry 正交

```
                    Industry（行业维度）
                    ┌────────┬────────┬────────┐
                    │ 零售   │ 金融   │ 制造   │
   ┌──────────────┼────────┼────────┼────────┤
C  │ knowledge.q  │  ✓     │  ✓     │  ✓     │  ← 能力跨行业
a  ├──────────────┼────────┼────────┼────────┤
p  │ workflow.ex  │  ✓     │  ✓     │  ✓     │
b  ├──────────────┼────────┼────────┼────────┤
i  │ vision.*     │  ✓     │  -     │  ✓     │  ← 部分不适用
l  └──────────────┴────────┴────────┴────────┘
t（能力维度）
```

**硬约束**：Capability ID 禁止行业词！

## ADR-004 §8：Connector 定位

> Connector 是受治理集成资源，**只在 SkillRelease execution context 内可用**。

| 约束 | 说明 |
|------|------|
| 不可独立调用 | Channel/Gateway/Runtime 都不能直接调 |
| P0 只读 | effect_policy=read_only |
| 单跳委托 | 不允许链式调用 |

In [ ]:
# Capability × Industry 正交矩阵可视化
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 6))

capabilities = ['knowledge.query', 'workflow.execute', 'vision.detect', 'agent.call']
industries = ['零售', '金融', '制造', '政企', '医疗']

# 1=适用, 0.3=部分适用, 0=不适用
matrix = np.array([
    [1, 1, 1, 1, 1],   # knowledge.query
    [1, 1, 1, 1, 1],   # workflow.execute
    [1, 0, 1, 0, 1],   # vision.detect
    [1, 1, 1, 1, 1],   # agent.call
])

colors = np.where(matrix == 1, '#2196F3', np.where(matrix == 0.3, '#FFC107', '#E0E0E0'))

for i in range(len(capabilities)):
    for j in range(len(industries)):
        rect = plt.Rectangle((j, len(capabilities)-1-i), 1, 1,
                             facecolor=colors[i][j], edgecolor='white', linewidth=2)
        ax.add_patch(rect)
        symbol = '✓' if matrix[i][j] == 1 else ('~' if matrix[i][j] == 0.3 else '—')
        ax.text(j + 0.5, len(capabilities) - 1 - i + 0.5, symbol,
                ha='center', va='center', fontsize=18, fontweight='bold',
                color='white' if matrix[i][j] == 1 else '#999')

ax.set_xlim(0, len(industries))
ax.set_ylim(0, len(capabilities))
ax.set_xticks([j + 0.5 for j in range(len(industries))])
ax.set_xticklabels(industries, fontsize=13)
ax.set_yticks([len(capabilities) - 1 - i + 0.5 for i in range(len(capabilities))])
ax.set_yticklabels(capabilities, fontsize=12)

ax.set_title('Capability × Industry 正交矩阵\n（ADR-003 冻结模型）', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Industry（行业维度）', fontsize=13)
ax.set_ylabel('Capability（能力维度）', fontsize=13)

# 添加图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='适用'),
    Patch(facecolor='#E0E0E0', label='不适用（空 cell ≠ 缺陷）'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)

plt.tight_layout()
plt.savefig('w8d5_capability_industry_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存：Capability × Industry 正交矩阵")

# 📚 Part 3：当前代码实现

## Capability 层（`capability/catalog.py`）

### CapabilityDescriptor 核心字段

```python
class CapabilityDescriptor(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)  # 冻结！
    
    capability_id: str          # 如 "langchat.knowledge.query"
    capability_version: str     # 如 "v1"
    lifecycle: "draft" | "published" | "deprecated"
    effects: "read" | "write" | "destructive"
    required_scopes: list       # 如 ["capability:knowledge:read"]
    approval_policy: "none" | "runtime_human_approval"
    runtime_binding: dict       # E6 后统一为 {}
```

### 不可变性保障

published 后以下字段**永久不可修改**：
- input_schema, output_schema, execution_mode
- effects, required_scopes, approval_policy
- provider, runtime_binding

要变更？只能 **废弃旧版本 → 发布新版本**。

## SkillRelease 层（`skill_release/`）

### W09 绑定示例

```python
_w09_descriptor = SkillReleaseDescriptor(
    skill_id="langchat.w09.internal.service",
    version="v1",
    effect_policy="read_only",      # P0 只读
    human_review_gate="conditional", # 敏感走人审
    workflow_binding={"workflow_id": "mall-internal-service"},
)
```

## Read-Only 守卫（`canonical/read_only_guard.py`）

```python
_WRITE_INDICATORS = frozenset({
    "http_request", "db_write", "tool_call", "provider_conditional_write"
})

def enforce_read_only(descriptor):
    if descriptor.effect_policy != "read_only":
        raise ReadOnlyViolationError(...)
    _check_workflow_binding_writes(descriptor)  # 递归扫描！
```

> 这是运行时守卫，不是配置校验。每次执行都检查！

In [ ]:
# 模拟 Capability 不可变性校验
from pydantic import BaseModel, ConfigDict, Field, field_validator
from typing import Literal
import re

_capability_id_re = re.compile(r"^[a-z0-9]+(\.[a-z0-9]+)*$")

class SimpleCapabilityDescriptor(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)
    capability_id: str
    version: str
    lifecycle: Literal["draft", "published", "deprecated"] = "published"
    effects: Literal["read", "write", "destructive"] = "read"
    provider: str = "langchat"

    @field_validator("capability_id")
    @classmethod
    def validate_id(cls, v):
        if not _capability_id_re.match(v):
            raise ValueError(f"capability_id 不合规: {v!r}\n"
                           f"  必须匹配: ^[a-z0-9]+(\.[a-z0-9]+)*$\n"
                           f"  禁止行业词、大写字母、下划线")
        return v

# 测试：合规的 Capability ID
valid_ids = [
    "langchat.knowledge.query",
    "langchat.workflow.execute", 
    "langchat.vision.detect",
]

print("=== 合规的 Capability ID ===")
for cid in valid_ids:
    try:
        cap = SimpleCapabilityDescriptor(capability_id=cid, version="v1")
        print(f"  ✅ {cid}")
    except Exception as e:
        print(f"  ❌ {cid}: {e}")

# 测试：违规的 Capability ID
invalid_ids = [
    ("langchat.retail.knowledge", "包含行业词 'retail'"),
    ("LangChat.Knowledge.Query", "包含大写字母"),
    ("langchat.finance.risk", "包含行业词 'finance'"),
]

print("\n=== 违规的 Capability ID ===")
for cid, reason in invalid_ids:
    try:
        cap = SimpleCapabilityDescriptor(capability_id=cid, version="v1")
        print(f"  ⚠️  {cid} — 未被拦截！")
    except Exception:
        print(f"  🚫 {cid} — 已拦截（{reason}）")

print("\n结论：ADR-003 正交硬约束在代码中强制执行")

In [ ]:
# 模拟 Read-Only 守卫的写指标检测
_WRITE_INDICATORS = frozenset({
    "http_request", "db_write", "tool_call", "provider_conditional_write"
})

def scan_for_writes(obj, path="root", depth=0):
    """递归扫描 workflow_binding 中的写指标"""
    findings = []
    if depth > 8:
        return findings
    if isinstance(obj, dict):
        for key, value in obj.items():
            skey = str(key).lower()
            if skey in _WRITE_INDICATORS:
                findings.append(f"  ⚠️  发现写指标 '{key}' @ {path}")
            if isinstance(value, str) and value.lower() in _WRITE_INDICATORS:
                findings.append(f"  ⚠️  发现写指标值 '{value}' @ {path}.{key}")
            findings.extend(scan_for_writes(value, f"{path}.{key}", depth+1))
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            findings.extend(scan_for_writes(item, f"{path}[{i}]", depth+1))
    return findings

# 测试用例
test_bindings = {
    "安全（纯只读）": {
        "workflow_id": "mall-internal-service",
        "schema_version": "v1",
        "nodes": [
            {"type": "llm_call", "model": "gpt-4"},
            {"type": "rag_search", "kb": "internal-rules"}
        ]
    },
    "危险（含 HTTP 请求）": {
        "workflow_id": "crm-sync",
        "nodes": [
            {"type": "llm_call"},
            {"type": "http_request", "url": "https://crm.example.com/api/write"}
        ]
    },
    "危险（含 DB 写）": {
        "workflow_id": "data-pipeline",
        "nodes": [
            {"type": "db_write", "table": "orders"}
        ]
    }
}

for name, binding in test_bindings.items():
    print(f"\n=== 测试：{name} ===")
    print(f"  workflow_binding: {binding}")
    findings = scan_for_writes(binding)
    if findings:
        for f in findings:
            print(f)
        print("  🔴 结果：被 read_only 守卫阻断！")
    else:
        print("  🟢 结果：通过 read_only 守卫检查")

# 📚 Part 4：完整执行链路

```
Agent Host
    │ POST /v1/skill-releases/{skill_id}/invoke
    │ Authorization: Bearer <lc_ service_agent key>
    │ + 六维上下文 headers
    ▼
SkillRelease Canonical Execute（execution_service.py）
    │ 1️⃣ prepare：校验身份、解析 descriptor
    │ 2️⃣ 幂等检查
    │ 3️⃣ HITL 检查
    │ 4️⃣ 创建执行记录（含六维上下文）
    │ 5️⃣ read_only 守卫
    │ 6️⃣ 分发执行 → executor_fn
    ▼
SkillRelease Executor（如 w09_invoke）
    │ 执行 Workflow（内部执行表示）
    │ Workflow 可能引用 MCP Connector
    ▼
MCP Connector（mcp_connection_model.py）
    │ stdio 或 sse 连接
    ▼
外部系统（知识库、ERP、CRM...）
```

> **Capability API 独立于这条链路——只提供元数据查询，不参与执行。**

In [ ]:
# 完整执行链路可视化
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# 链路节点
nodes = [
    (5, 9, 'Agent Host\n（OpenClaw / 渠道 Agent）', '#E3F2FD', '#1976D2'),
    (5, 7.5, 'SkillRelease Canonical Execute\n（execution_service.py）', '#FFF3E0', '#F57C00'),
    (2, 5.5, '1️⃣ 准备：校验身份\n2️⃣ 幂等检查\n3️⃣ HITL 检查', '#E8F5E9', '#388E3C'),
    (5, 5.5, '4️⃣ 创建执行记录\n（六维上下文）\n5️⃣ read_only 守卫', '#E8F5E9', '#388E3C'),
    (8, 5.5, '6️⃣ 分发执行\n→ executor_fn', '#E8F5E9', '#388E3C'),
    (5, 3.5, 'SkillRelease Executor\n（w09_invoke 等）', '#F3E5F5', '#7B1FA2'),
    (2.5, 1.8, 'MCP Connector\n（stdio / sse）', '#FFEBEE', '#C62828'),
    (7.5, 1.8, '外部系统\n（KB / ERP / CRM）', '#ECEFF1', '#455A64'),
]

for x, y, label, bg, border in nodes:
    box = mpatches.FancyBboxPatch((x-1.5, y-0.55), 3, 1.1,
                                   boxstyle="round,pad=0.15",
                                   facecolor=bg, edgecolor=border, linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=9.5, fontweight='bold')

# 箭头
arrows = [
    (5, 8.45, 5, 8.05, 'POST /v1/skill-releases/{id}/invoke'),
    (5, 6.95, 5, 6.55, '执行'),
    (3.5, 5.5, 3.5, 5.5, ''),
    (5, 4.95, 5, 4.05, ''),
    (5, 2.95, 5, 2.35, '调用'),
    (3.5, 1.8, 6, 1.8, '连接'),
]

ax.annotate('', xy=(5, 8.05), xytext=(5, 8.45),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))
ax.text(5.3, 8.25, 'invoke', fontsize=8, color='#666')

ax.annotate('', xy=(5, 6.05), xytext=(5, 6.95),
            arrowprops=dict(arrowstyle='->', color='#F57C00', lw=2))
ax.text(5.3, 6.5, 'canonical\nexecute', fontsize=8, color='#F57C00', ha='left')

ax.annotate('', xy=(5, 4.05), xytext=(5, 4.95),
            arrowprops=dict(arrowstyle='->', color='#388E3C', lw=2))
ax.text(5.3, 4.5, 'dispatch', fontsize=8, color='#388E3C')

ax.annotate('', xy=(5, 2.35), xytext=(5, 2.95),
            arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=2))
ax.text(5.3, 2.65, 'executor', fontsize=8, color='#7B1FA2')

ax.annotate('', xy=(6, 1.8), xytext=(4, 1.8),
            arrowprops=dict(arrowstyle='->', color='#C62828', lw=2))
ax.text(5, 2.0, 'connect', fontsize=8, color='#C62828', ha='center')

# 旁路：Capability API
cap_box = mpatches.FancyBboxPatch((8, 7), 2.5, 1,
                                   boxstyle="round,pad=0.1",
                                   facecolor='#FFF9C4', edgecolor='#F9A825', linewidth=1.5,
                                   linestyle='--')
ax.add_patch(cap_box)
ax.text(9.25, 7.5, 'Capability API\n（仅元数据查询）\n不参与执行',
        ha='center', va='center', fontsize=8, color='#F57C00', style='italic')

ax.set_title('LangChat 执行链路：Capability → Connector → 企业系统\n（W8-Day5 完整链路图）',
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('w8d5_execution_chain.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存：完整执行链路图")

# 📊 Part 5：Gap Analysis

| 维度 | ADR 目标态 | 当前代码现实 | Gap |
|------|-----------|-------------|-----|
| Capability 执行 | 不直接执行 | ✅ `/invoke` 已移除 | 已对齐 |
| Capability 发现 | 元数据查询 | ✅ list + describe | 已对齐 |
| 正交模型 | ID 禁行业词 | ✅ 正则校验 | 已对齐 |
| SkillRelease 唯一入口 | 所有执行经 canonical | ⚠️ E3/E4 遗留 | P1 |
| Connector 治理 | 独立登记+审计 | ❌ 嵌在 Workflow 内 | **最大 Gap** |
| read-only 强制 | 运行时守卫 | ✅ enforce_read_only | 已对齐 |
| 六维上下文 | 全链路传递 | ✅ Principal 携带 | 已对齐 |

## 🔴 最大 Gap：Connector 治理缺失

当前 MCP Connector 嵌在 Workflow 内部：
1. Connector 调用不经独立 Capability Resolution
2. Connector 没有独立 effect_policy 校验
3. Connector 版本管理依赖 Workflow
4. Connector 未在 Platform Governance Plane 独立登记

# 💡 今天多理解了什么

| # | 以前以为 | 现在知道 |
|---|---------|---------|
| 1 | Capability = Plugin 新名字 | Capability 是治理描述符，不包含执行逻辑 |
| 2 | Capability 和 SkillRelease 同层 | 三层：Capability（描述）→ SkillRelease（执行）→ Workflow（实现）|
| 3 | Connector = API Gateway | Connector 是受治理集成资源，只在 execution context 内可用 |
| 4 | `/invoke` 还在 | E6 migration 已完全移除 |
| 5 | runtime_binding 指向执行器 | E6 后统一为 `{}` |
| 6 | read_only 是配置项 | 是运行时守卫！递归扫描 workflow_binding |
| 7 | 行业属性写在 Capability 里 | ADR-003 明确禁止，正交维度 |
| 8 | P0 有 Connector 治理 | 还没有，MCP 嵌在 Workflow 内是最大 Gap |

# 🔮 重新设计还会这样做吗？

**会，而且更早分开。**

1. **职责分离不可妥协**：合并会导致 N 份重复描述或超级描述符
2. **Connector 独立治理应更早做**：当前 MCP 嵌在 Workflow 内是最大架构债
3. **E6 证明"Capability 不执行"是正确的**：如果一开始就不给执行能力，E6 不需要存在

**不会改变的**：正交模型、不可变描述符、read_only 运行时守卫、六维上下文

# 📝 Daily Engineering Log

### 新增
- Capability 是治理描述层，E6 后 `runtime_binding={}`
- `enforce_read_only()` 递归扫描 workflow_binding 查找写指标
- SkillReleaseDescriptor 三个 model_validator 约束

### 修改
- Capability API 不是"受限"而是"完全移除了执行"
- Connector ≠ Gateway
- runtime_binding 已统一为空

### 确认
- Capability × Industry 正交已在代码强制执行
- P0 两个预置 Capability effects 都是 read
- Canonical Execution Service 六步流程可追踪

### 技术债
- MCP Connector 未独立治理
- `_WRITE_INDICATORS` 枚举式检测有覆盖盲区
- Connector 版本管理依赖 Workflow

### 下一步
- Day6：画完整链路图（用户→SkillRelease→Workflow→Connector→企业系统）
- Day7：Virtual CTO Review

# 📖 术语表

| 英文 | 音标 | 中文 |
|------|------|------|
| Capability | /ˌkeɪpəˈbɪləti/ | 能力 |
| CapabilityDescriptor | /dɪˈskrɪptər/ | 能力描述符 |
| Connector | /kəˈnektər/ | 连接器 |
| SkillRelease | /skɪl rɪˈliːs/ | 技能发布 |
| Plugin | /ˈplʌɡɪn/ | 插件 |
| Effect Policy | /ɪˈfekt ˈpɒləsi/ | 效果策略 |
| Human Review Gate | /hjuːmən rɪˈvjuː ɡeɪt/ | 人审门控 |
| Read-Only Guard | /riːdˈəʊnli ɡɑːd/ | 只读守卫 |
| Orthogonal Facet | /ɔːˈθɒɡənəl ˈfæsɪt/ | 正交维度 |
| MCP | /em-siː-piː/ | 模型上下文协议 |
| Canonical | /kəˈnɒnɪkəl/ | 规范的 |
| Idempotency | /ˌaɪdəmˈpɒtənsi/ | 幂等性 |

# ❓ 课堂练习

**练习 1**：对比 Capability 和 Plugin 在治理、版本、安全、执行、组合、审计六个维度的区别。

**练习 2**：追踪 Agent Host 调用 `langchat.w09.internal.service` 的完整链路（6步）。

**练习 3**：判断以下 Capability ID 是否合规：
- `langchat.retail.knowledge.query`
- `langchat.vision.detect`
- `LangChat.Knowledge.Query`

---

# 📝 课后测试

**Q1**：为什么 SkillRelease 是对外契约而不是 Workflow？
- A. Workflow 不稳定
- B. Workflow 是内部实现细节，格式可能替换 ✅

**Q2**：`enforce_read_only()` 检查哪些内容？（多选）
- A. effect_policy ✅ B. workflow_binding 写指标 ✅

**Q3**：Connector 在什么条件下可调用外部系统？
- B. 只在 SkillRelease 已授权 execution context 内 ✅

---

# 📚 真实参考

- ADR-001 §7 分类法、§6 控制面/执行面
- ADR-003 (docs) §2.2 正交硬约束
- ADR-004 §4.1 唯一执行入口、§8 Connector 边界
- `capability/catalog.py` — CapabilityDescriptor + CapabilityRegistry
- `skill_release/descriptor.py` — SkillReleaseDescriptor
- `skill_release/canonical/execution_service.py` — CanonicalExecutionService
- `skill_release/canonical/read_only_guard.py` — 只读守卫
- `docs/api/capability-api.md` — API 文档